# Demo **Paraphrasing**

Our imposter approach needs models to reword an original text keeping the same meaning.
Ideally, we want to model the way texts originated.
This is dependent on the text's genre, i.e. a student essay originates from a task description and a list of URLs while a new article originates from a list of additional sources or from spectating an event.
We implemented both a naive approach to rewording a text and a more sophisticated one:
- given a prompt and a text, the naive approach uses a language model to paraphrase the text
- given a prompt and a text LLM A is used to extract the text's genre, tone and summary of bullet points, then LLM B is used to reword the text in the same genre and tone based on the summary of bullet points

In [2]:
from pathlib import Path
import os
import textwrap
import sys

# Add the root of the project to Python path
sys.path.append(os.path.abspath(".."))
from genai_detection.config import CONFIG
from genai_detection.paraphrasing.paraphraser import (
    T5ChatGPTParaphraser,
    T5GooglePAWSParaphraser,
    TranslationParaphraser,
    BulletPointParaphraser,
    OllamaParaphraser,
    TaskParaphraser,
    TopicParaphraser,
    TitleParaphraser,
    SAIAParaphraser,
)

## Original text
We use a CNN news article from 23.06.2025/ 04.07.2025 as an example.

In [3]:
path2datasets = (
    Path(os.getcwd()).resolve().parent / "data" / "datasets" / "custom_texts"
)
data_category = "News"  # This is used for plot titles in the notebook
path2results = Path(os.getcwd()).resolve().parent / CONFIG.SAVE_PATH / "paraphrasing"
assert path2datasets.exists(), f"Path to datasets {path2datasets} does not exist."
# file_name = "cnn_230625"    # USA attacks Iran
file_name = "cnn_040725"  # Dalai Lama
original_text = open(path2datasets / f"{file_name}.txt").read()

print(textwrap.fill(original_text, width=80))

  For much of the past century, the Dalai Lama has been the living embodiment of
Tibet’s struggle for greater freedoms under Chinese Communist Party rule,
sustaining the cause from exile even as an increasingly powerful Beijing has
become ever more assertive in suppressing it.  As his 90th birthday approaches
this Sunday, the spiritual leader for millions of followers of Tibetan Buddhism
worldwide is bracing for a final showdown with Beijing: the battle over who will
control his reincarnation.  On Wednesday, the Dalai Lama announced that he will
have a successor after his death, and that his office will have the sole
authority to identify his reincarnation.  “I am affirming that the institution
of the Dalai Lama will continue,” the Nobel Peace laureate said in a video
message to religious elders gathering in Dharamshala, India, where he has found
refuge since Chinese communist troops put down an armed uprising in his
mountainous homeland in 1959.  The cycle of rebirth lies at the core 

## Naive Approach

In [4]:
paraphrasers = {
    "T5_ChatGPT": T5ChatGPTParaphraser(),
    "T5_Google_PAWS": T5GooglePAWSParaphraser(),
    "Ollama": OllamaParaphraser(model_id=CONFIG.OLLAMA_MODEL),
    "SAIA": SAIAParaphraser(model_id=CONFIG.SAIA_MODEL),
}
if "Blablador" in paraphrasers.keys():
    print(
        "Available Blablador models:\n",
        paraphrasers["Blablador"]._get_available_models(verbose=False),
    )

Using T5ChatGPTParaphraser
Loaded T5ChatGPTParaphraser tokenizer
Loaded T5ChatGPTParaphraser model
Using T5GooglePAWSParaphraser


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Loaded T5GooglePAWSParaphraser tokenizer
Loaded T5GooglePAWSParaphraser model


In [ ]:
n_responses = 1  # number of paraphrases to generate
max_length = 512  # Maximum length of the generated paraphrase
temperature = 0.7  # Controls the randomness of the output. Lower values make the output more deterministic.

prompts = [
    "Paraphrase the text above without changing its meaning. Use different words and vary the sentence structure while maintaining a consistent tone. Your paraphrase should be three times as long than the original. Output only the paraphrased sentence, with NO explanations or extra text.",
    "Paraphrase the given sentence by identifying the main subject, verb, and object. Replace each with synonyms or closely related words, adjusting grammar naturally. Keep the new sentence close in length to the original. Output only the final paraphrased sentence.",
    "Paraphrase the sentence above without changing its meaning. Use different words and vary the sentence structure while keeping the tone consistent. Keep the new sentence similar in length to the original. Output only the paraphrased sentence, with no explanations or extra text.",
]

In [6]:
for paraphraser_name, paraphraser in paraphrasers.items():
    print(f"\n{'='*40}")
    print(f"\nUsing paraphraser: {paraphraser_name}")
    for prompt in prompts:
        print(f"\nPrompt: {prompt}")
        paraphrases = paraphraser.paraphrase(
            original_text,
            n_responses=n_responses,
            max_length=max_length,
            temperature=temperature,
            prompt=prompt,
        )
        for i, paraphrase in enumerate(paraphrases):
            print(f"Paraphrase {i+1}: {textwrap.fill(paraphrase, width=80)}")



Using paraphraser: T5_ChatGPT

Prompt: Paraphrase the text above and output only the paraphrased version.
T5 model


Group Beam Search is scheduled to be moved to a `custom_generate` repository in v4.55.0. To prevent loss of backward compatibility, add `trust_remote_code=True` to your `generate` call.


KeyboardInterrupt: 

## Bullet Point Approach

Ollama models are by far the best ones for extracting the genre, tone and summary of bullet points and returning them in a structured way.
Hence, we use the Ollama model as the text extractor and any other model as the text generator.

In [ ]:
models = {
    "text_extractor": {
        "name": "Ollama-latest",
        "model": SAIAParaphraser(model_id=CONFIG.SAIAI_VERSION),
    },
    "text_generator": {
        "name": "Ollama-latest",
        "model": SAIAParaphraser(model_id=CONFIG.SAIAI_VERSION),
    },
}
bullet_point_paraphraser = BulletPointParaphraser(
    text_extractor=models["text_extractor"]["model"],
    text_generator=models["text_generator"]["model"],
)
paraphrases = bullet_point_paraphraser.paraphrase(
    text=original_text, prompt=None, n_responses=n_responses, max_length=max_length
)
for i, paraphrase in enumerate(paraphrases):
    print(f"Paraphrase {i+1}:\n{paraphrase}\n")
# save to file
# with open(path2datasets / f"paraphrase_{file_name}_e={models["text_extractor"]['name']}_g={models["text_generator"]['name']}_{i+1}.txt", "w") as f:
#     f.write(paraphrase)

BulletPoint model
SAIA model
SAIA model
Paraphrase 1:
The death of the 14th Dalai Lama on 90 April 2025 marks the end of an era that has shaped both the spiritual life of Tibetan Buddhism and the geopolitical calculus of the Himalayan region for more than six decades. In a carefully prepared statement released by his office, the late spiritual leader announced a succession plan that diverges sharply from the expectations of the People’s Republic of China. The plan stipulates that his next incarnation will be identified by a committee of senior Tibetan monks in exile, an assembly that will operate independently of any state authority. The announcement, delivered just days before the Dalai Lama’s passing, underscores a deliberate effort to preserve the religious autonomy of the tradition and to forestall what he described as “the intrusion of external political control into a sacred process.”

The Dalai Lama’s appeal to the Tibetan diaspora and to the broader international community was 

## Task Paraphraser
The task paraphraser extracts the task, tone and genre from the text and then generates a paraphrase based on this information.

In [ ]:
task_paraphraser = TaskParaphraser(
    text_extractor=models["text_extractor"]["model"],
    text_generator=models["text_generator"]["model"],
)
paraphrases = task_paraphraser.paraphrase(
    text=original_text, prompt=None, n_responses=n_responses, max_length=max_length
)
for i, paraphrase in enumerate(paraphrases):
    print(f"Paraphrase {i+1}:\n{paraphrase}\n")

Task model
SAIA model
SAIA model
Paraphrase 1:
**Lhasa, 9 August 2025** – As the 89‑year‑old 14th Dalai Lama approaches the final months of his life, the question of his successor has moved from a theological discussion to a flashpoint of regional geopolitics. The Tibetan exile community, led by the Central Tibetan Administration (CTA) in Dharamshala, insists that the reincarnation process must be conducted according to centuries‑old religious protocols and free from state interference. Beijing, meanwhile, has repeatedly asserted its sovereign right to approve any future Dalai Lama, framing the matter as a test of its authority over “internal affairs.” The clash over the next incarnation is now intertwined with broader strategic calculations involving China’s border security, India’s regional ambitions, and the fragile human‑rights situation in the Himalayas.

The Dalai Lama’s death would trigger a traditional search for the “tulku,” a child identified through a combination of visions,

## Topic Paraphraser
The topic paraphraser extracts the topic, tone and genre from the text and then generates a paraphrase based on this information.

In [ ]:
topic_paraphraser = TopicParaphraser(
    text_extractor=models["text_extractor"]["model"],
    text_generator=models["text_generator"]["model"],
)
paraphrases = topic_paraphraser.paraphrase(
    text=original_text, prompt=None, n_responses=n_responses, max_length=max_length
)
for i, paraphrase in enumerate(paraphrases):
    print(f"Paraphrase {i+1}:\n{paraphrase}\n")

Topic model
SAIA model
SAIA model
Paraphrase 1:
The question of who will succeed the 14th Dalai Lama has resurfaced as a geopolitical flashpoint, drawing the Tibetan exile community, the Chinese state, and the broader international community into a contest that intertwines religious tradition, national sovereignty, and human‑rights diplomacy. Since the Dalai Lama’s public declaration in 2011 that his reincarnation might be found outside Tibet, Beijing has accelerated its own parallel search, invoking statutes that claim exclusive authority over the identification of high‑lamas. The clash is no longer a purely spiritual matter; it has become a strategic arena where Beijing seeks to reinforce its territorial claims, while the Central Tibetan Administration (CTA) strives to preserve the integrity of Tibetan Buddhism and the political legitimacy of the exile government.

The historical background is essential for understanding the stakes. The Dalai Lama, traditionally recognized through a 

## Title Paraphraser
The title paraphraser extracts the title, tone and genre from the text and then generates a paraphrase based on this information.

In [ ]:
title_paraphraser = TitleParaphraser(
    text_extractor=models["text_extractor"]["model"],
    text_generator=models["text_generator"]["model"],
)
paraphrases = title_paraphraser.paraphrase(
    text=original_text, prompt=None, n_responses=n_responses, max_length=max_length
)
for i, paraphrase in enumerate(paraphrases):
    print(f"Paraphrase {i+1}:\n{paraphrase}\n")

Title model
SAIA model
SAIA model
Paraphrase 1:
The Dalai Lama’s Succession and the Sino‑Tibetan Power Struggle  

The question of who will recognize the next incarnation of the Dalai Lama has resurfaced as a flashpoint in the broader contest for influence between the People’s Republic of China and the Tibetan diaspora. While the 14th Dalai Lama, Tenzin Gyatso, remains alive and active, the mechanisms governing his eventual succession have become a proxy for competing narratives about sovereignty, religious autonomy, and human‑rights standards in the 2020s. For informed observers of geopolitics, the unfolding debate offers a rare window into how an ancient spiritual institution can be weaponised within contemporary statecraft.

The Dalai Lama occupies a unique position at the intersection of religion and politics. Historically, the institution combined spiritual leadership with temporal authority over the Tibetan plateau until the Chinese annexation in 1950. After the ensuing exile of 

# Translation Paraphraser

In [ ]:
translation_paraphraser = TranslationParaphraser(
    text_extractor=models["text_extractor"]["model"],
    text_generator=models["text_generator"]["model"],
    language="French",
)
paraphrases = translation_paraphraser.paraphrase(
    text=original_text, prompt=None, n_responses=n_responses, max_length=max_length
)
for i, paraphrase in enumerate(paraphrases):
    print(f"Paraphrase {i+1}:\n{paraphrase}\n")

[DEBUG] Using TranslationParaphraser with prompt: Translate the text above into French. Do not use direct quotes or newlines. Output only the translated text, without any additional commentary or formatting.
Paraphrase 1:




## Batch Processing

In [ ]:
# Batch paraphrasing
texts = [
    "The sky is blue.",
    "I love programming.",
    "Artificial intelligence is fascinating.",
]
paraphrased_batch = bullet_point_paraphraser.paraphrase_batch(texts)
for original, paraphrased in zip(texts, paraphrased_batch):
    print(f"Original: {original} | Paraphrased: {paraphrased}")

BulletPoint model
SAIA model
SAIA model
BulletPoint model
SAIA model
SAIA model
BulletPoint model
SAIA model
SAIA model
Original: The sky is blue. | Paraphrased: ['Blue sky from scattering', 'Daytime blue sky scattering', 'Daytime sky appears blue', 'Blue sky from scattering', 'Midday blue sky appears']
Original: I love programming. | Paraphrased: ['Coding fuels me!', 'Passionate about coding!', 'Code, create, conquer!', 'I love coding!', 'Coding, creating, solving!']
Original: Artificial intelligence is fascinating. | Paraphrased: ['AI exploding, shaping everything!', 'AI and ML explode!', 'AI fuels future breakthroughs!', 'AI rockets, sparking debates!', 'AI fuels rapid breakthroughs!']
